# Bnchmarks (Anomalib)
Runs Anomalib benchmarks on Folder datasets. Point ROOTS to dataset where concrete defect type datasets are located dirs; we resolve the inner dataset root.
Defects are encoded in filenames in test/bad between __ __.

In [ ]:
# ===================== 1) IMPORTS =====================
import os
os.environ["TQDM_DISABLE"] = "1" # Othervise some algorithm may cause recursion error in Jupyter notebook

import gc
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

tqdm.disable = True

try:
    from anomalib.models import Patchcore, Padim, Dfm, Cfa, Stfpm
    from anomalib.engine import Engine
    from anomalib.data import Folder
except Exception as e:
    raise ImportError(
        "Anomalib is not available. Install anomalib in your environment to run benchmarks.\n" + str(e)
    )


In [ ]:
# ===================== 2) CONSTANTS / PATHS (edit here) =====================
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_BASE = Path("./").resolve()
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
ROOTS = {
    "circulation_quality_public": Path("./dataset_public_circulation_quality").resolve(),
    "manufacturing_defects_public": Path("./dataset_public_manufacturing_defects").resolve(),
}

IMAGE_SIZE = [128, 128]
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
NUM_RUNS = 1  # Increase for more reliable statistics

RESULTS_FILE = OUTPUT_BASE / "anomalib_results_by_dataset_benchmark.csv"
RESULTS_PER_DEFECT_FILE = OUTPUT_BASE / "anomalib_results_per_defect_benchmark.csv"
TEMP_BASE = Path("/tmp/anomalib_filtered_dataset")

for k, p in ROOTS.items():
    print(k, '->', p)
    assert p.exists(), f"Dataset root not found: {p}"

print('OUTPUT_BASE:', OUTPUT_BASE)


In [ ]:
# ===================== 3) Helpers =====================

def resolve_dataset_root(root: Path) -> Path:
    root = Path(root)
    for cand in root.rglob('*'):
        if cand.is_dir() and (cand / 'train').is_dir() and (cand / 'test').is_dir():
            return cand
    raise FileNotFoundError(f"Could not find inner dataset root with train/ and test/ under {root}")


def parse_defect_from_filename(filename: str, unknown='unknown') -> str:
    m = re.search(r'__([^_].*?)__', filename)
    return m.group(1) if m else unknown


def build_folder_datamodule(name: str, root: Path) -> Folder:
    dm = Folder(
        name=name,
        root=str(root),
        normal_dir="train/good",
        abnormal_dir="test/bad",
        normal_test_dir="test/good",
        train_batch_size=TRAIN_BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        image_size=IMAGE_SIZE,
        seed=SEED,
        task="classification",
    )
    dm.setup()
    return dm


def create_models():
    return {
        "Patchcore": Patchcore(
            backbone="resnet18",
            coreset_sampling_ratio=0.001, #Adjust depending on available GPU memory
            pre_trained=True,
        ),
        "PaDiM": Padim(backbone="resnet18"),
        "DFM": Dfm(backbone="resnet18"),
        "CFA": Cfa(backbone="resnet18"),
        "STFPM": Stfpm(backbone="resnet18"),
    }


In [ ]:
# ===================== 4) Benchmark per dataset =====================
from tqdm import tqdm
tqdm.disable = True   # Otherwise will cause recursion errors in CFA


def run_one(engine: Engine, model, datamodule: Folder):
    engine.fit(model=model, datamodule=datamodule)
    return engine.test(model=model, datamodule=datamodule)


def benchmark_datasets(roots: dict) -> pd.DataFrame:
    rows = []
    for ds_key, root in roots.items():
        dataset_root = resolve_dataset_root(root)
        dm = build_folder_datamodule(ds_key, dataset_root)

        for run in range(NUM_RUNS):
            models = create_models()
            for model_name, model in models.items():
                engine = Engine(max_epochs=1,task="classification", image_metrics=["AUROC",  "F1Score"],
                                logger=False)
                print(f"Dataset={ds_key} | Model={model_name} | Run={run+1}/{NUM_RUNS}")

                result = run_one(engine, model, dm)[0]
        
                row = {"dataset": ds_key, "model": model_name, "run": run}
                if isinstance(result, dict):
                    row.update(result)
                rows.append(row)
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    return pd.DataFrame(rows)


results_df = benchmark_datasets(ROOTS)
results_df.to_csv(RESULTS_FILE, index=False)
print('Saved:', RESULTS_FILE)
display(results_df.head())


## Per-defect benchmarking


In [ ]:
# ===================== 5) Per-defect benchmarking (optional) =====================

def list_defects_in_bad(dataset_root: Path) -> list[str]:
    bad_root = dataset_root / 'test' / 'bad'
    if not bad_root.exists():
        return []
    defects = []
    for f in bad_root.rglob('*'):
        if f.is_file():
            defects.append(parse_defect_from_filename(f.name))
    return sorted(set(defects))


def make_filtered_dataset_by_defect(src_root: Path, defect: str, dst_root: Path) -> Path:
    import shutil

    if dst_root.exists():
        shutil.rmtree(dst_root)

    (dst_root / 'train' / 'good').mkdir(parents=True, exist_ok=True)
    (dst_root / 'test' / 'good').mkdir(parents=True, exist_ok=True)
    (dst_root / 'test' / 'bad').mkdir(parents=True, exist_ok=True)

    shutil.copytree(src_root / 'train' / 'good', dst_root / 'train' / 'good', dirs_exist_ok=True)
    shutil.copytree(src_root / 'test' / 'good', dst_root / 'test' / 'good', dirs_exist_ok=True)

    src_bad = src_root / 'test' / 'bad'
    for f in src_bad.rglob('*'):
        if not f.is_file():
            continue
        if parse_defect_from_filename(f.name) != defect:
            continue
        shutil.copy2(f, dst_root / 'test' / 'bad' / f.name)

    return dst_root


def benchmark_per_defect(roots: dict) -> pd.DataFrame:
    rows = []

    for ds_key, root in roots.items():
        dataset_root = resolve_dataset_root(root)
        defects = list_defects_in_bad(dataset_root)
        if not defects:
            print(f"[WARN] No defects found in {dataset_root}/test/bad")
            continue

        for defect in defects:
            filtered_root = TEMP_BASE / f"{ds_key}__{defect}"
            filtered_root = make_filtered_dataset_by_defect(dataset_root, defect, filtered_root)
            dm = build_folder_datamodule(f"{ds_key}:{defect}", filtered_root)

            for run in range(NUM_RUNS):
                # Models and engine has to be recreated for each model every tun. otherwise it will cause memory leak and OOM in later runs.
                models = create_models()
                for model_name, model in models.items():
                    engine = Engine(max_epochs=1,task="classification", image_metrics=["AUROC",  "F1Score"],
                                logger=False)
                    print(f"Dataset={ds_key} | Defect={defect} | Model={model_name} | Run={run+1}/{NUM_RUNS}")

                    result = run_one(engine, model, dm)[0]
                    row = {"dataset": ds_key, "defect": defect, "model": model_name, "run": run}
                    if isinstance(result, dict):
                        row.update(result)
                    rows.append(row)
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


per_defect_df = benchmark_per_defect(ROOTS)
per_defect_df.to_csv(RESULTS_PER_DEFECT_FILE, index=False)
print('Saved:', RESULTS_PER_DEFECT_FILE)
display(per_defect_df.head())
